<a href="https://colab.research.google.com/github/banuben/computer_vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
print('Menu Detector!')

Menu Detector!


In [36]:
# -----------------
# Import Libraries
# -----------------


import torch
import torch.nn as nn
# import torch.optim as optim

from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

from torchvision import transforms    # this's working
import torchvision.transforms as transforms
# from torchvision.datasets import ImageFolder
# from torch.utils.data import DataLoader
from google.colab import drive
drive.mount('/content/drive')

from torch.utils.data import Dataset, DataLoader
from PIL import Image, UnidentifiedImageError
import os
import numpy as np




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
# Define Dataset Path

DATASET_PATH = '/content/drive/MyDrive/food101_dataset'
print('Dataset_path:',DATASET_PATH)


CUSTOM_CLASS_MAPPING = {
    'hamburger': 'hamburger',
    'hot_dog': 'hot_dog',
    'chocolate_cake': 'dessert',  # Label Grouping | class consolidation
    'cheesecake': 'dessert',
    'kebab': 'kebab',
    'pilaf': 'pilaf'
}


CLASSES = ['hamburger', 'hot_dog', 'dessert', 'kebab', 'pilaf']
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)
print('Classes:',CLASSES)
print('Class_to_box:',CLASS_TO_IDX)
print('Num_classes:',NUM_CLASSES)


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

Dataset_path: /content/drive/MyDrive/food101_dataset
Classes: ['hamburger', 'hot_dog', 'dessert', 'kebab', 'pilaf']
Class_to_box: {'hamburger': 0, 'hot_dog': 1, 'dessert': 2, 'kebab': 3, 'pilaf': 4}
Num_classes: 5


In [38]:
# --------------------
# Custom Dataset Class
# --------------------

class FoodDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        print('images_length', len(self.images))
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        print('image_path', img_path)

        label = self.labels[idx]
        print('label', label)

        try:
            image = Image.open(img_path).convert('RGB')
        except (UnidentifiedImageError, OSError):
            print(f"Skipping broken image: {img_path}")
            return self.__getitem__((idx + 1) % len(self))

        if self.transform:
            image = self.transform(image)

        return image, label

In [39]:
# Gather and Split Data

all_images = []

for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
    class_path = os.path.join(DATASET_PATH, original_class)
    print('class_path', class_path)

    if not os.path.exists(class_path):
        print('Warning: {class_path} not found')
        continue

    for img in os.listdir(class_path):
        if img.endswith(('.jpg', '.jpeg', '.png')):
            full_path = os.path.join(class_path, img)
            all_images.append((full_path, CLASS_TO_IDX[mapped_class]))


np.random.shuffle(all_images)

split = int(0.8 * len(all_images))

train_data = all_images[:split]
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

# print('all_images:', all_images)

dataset = FoodDataset(train_images, train_labels)
print(len(dataset))

img, lbl = dataset[0]

class_path /content/drive/MyDrive/food101_dataset/hamburger
class_path /content/drive/MyDrive/food101_dataset/hot_dog
class_path /content/drive/MyDrive/food101_dataset/chocolate_cake
class_path /content/drive/MyDrive/food101_dataset/cheesecake
class_path /content/drive/MyDrive/food101_dataset/kebab
class_path /content/drive/MyDrive/food101_dataset/pilaf
images_length 3233
3233
image_path /content/drive/MyDrive/food101_dataset/chocolate_cake/670249.jpg
label 2


In [40]:
train_dataset = FoodDataset(train_images, train_labels, transform=transform)
val_dataset = FoodDataset(val_images, val_labels, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)  # thread | parallel loading for speed
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)



images_length 3233
images_length 3233


In [41]:
# pretrained model

model = mobilenet_v2(
    weights=MobileNet_V2_Weights.IMAGENET1K_V1
)  # pretrained model | lightweight | CNN

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    NUM_CLASSES
)  # fine-tuning | backbone | model layer freeze

#

In [42]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
model = model.to(device)

Device: cuda


In [43]:
criterion = nn.CrossEntropyLoss()  # Loss Function | shows in percentage
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)  # weight
torch.backends.cudnn.benchmark = True  # Benchmark Setting | Trick | 10-20% faster

print('Criterion:', criterion)
print('Optimizer:', optimizer)

Criterion: CrossEntropyLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
